# SwasthyaSetu · 01 · PDF Inspection & Profiling

**Source:** *Guidelines on Human Resources for Health for NHM (2022)* — 174 pages
**Goal:** understand the PDF completely before extracting anything. Nothing is changed here — we only look, measure and decide.

| Stage | Steps | Question it answers |
|---|---|---|
| A · Inspect | 1–4 | What kind of pages does the PDF have? |
| B · Profile | 5–11 | What does the text look like, and what is clutter? |

**Output of this notebook:** a set of rules and page lists used by the cleaning notebook (Stage C).

## Step 1 · Basic file facts
**Why:** know what we're dealing with before choosing tools.
**Result:** 174 pages · A4 (595 × 842 pt) · made in InDesign → most pages have a real text layer; some pages are scanned images.
**Decision:** use pdfplumber for text pages, OCR for scanned pages.

In [53]:
#Stage A: Inspect (look before touching anything)

#opening the pdf file 

#load the Library 
import pdfplumber

pdf_path = "../data/raw/Final Guideline on Human Resources for Health for NHM.pdf"

#open the file
with pdfplumber.open(pdf_path) as nhm_pdf:
    print("Total number of pages :",len(nhm_pdf.pages))
    metadata = nhm_pdf.metadata
    print("Creator software:", metadata.get("Creator"))
    print("Producer software:", metadata.get("Producer"))
    for page in nhm_pdf.pages[:20]:
        print(f"Page no:' {page} has text layer {len(page.chars) > 0} and  has image layer {len(page.images) > 0}")
   

Total number of pages : 174
Creator software: Adobe InDesign 16.0 (Windows)
Producer software: Adobe PDF Library 15.0
Page no:' <Page:1> has text layer False and  has image layer True
Page no:' <Page:2> has text layer False and  has image layer False
Page no:' <Page:3> has text layer True and  has image layer False
Page no:' <Page:4> has text layer False and  has image layer False
Page no:' <Page:5> has text layer False and  has image layer True
Page no:' <Page:6> has text layer False and  has image layer False
Page no:' <Page:7> has text layer False and  has image layer True
Page no:' <Page:8> has text layer False and  has image layer False
Page no:' <Page:9> has text layer False and  has image layer True
Page no:' <Page:10> has text layer False and  has image layer False
Page no:' <Page:11> has text layer False and  has image layer True
Page no:' <Page:12> has text layer False and  has image layer False
Page no:' <Page:13> has text layer False and  has image layer True
Page no:' <Pag

It contain both image and text so need OCR 

## Step 2 · Page-type map — `classify_pdf_pages()`
**Why:** different page types need different handling (text, OCR, skip).
**Input → Output:** PDF → one label per page (+ `part` = chapter name)
**How:**
- Main signal = the **rotated side label** in the margin (e.g. `RECRUITMENT`); reversed on left pages → `text[::-1]`
- Pages without a label → decided by chars / images / lines
- Form pages come from a manual list

**Result:** chapter body 29 (PDF 21–49) · annexure procedures 45 · forms 14 · scanned letters 37 · case studies 24 · scanned messages 5 · contents 4 · abbreviations 2 · notes 3 · blank 8 · cover/title 3
**Decision:** `part` column becomes chunk metadata later.

In [117]:
import pdfplumber
import pandas as pd

# Side labels printed sideways in the page margin (clean spelling)
PART_LABELS = {
    "LIST OF ABBREVIATIONS":         "abbreviations",
    "CONTENTS":                      "contents",
    "SUGGESTIVE PROCEDURAL FORMATS": "annexure I (procedures)",
    "COMPENDIUM OF LETTERS":         "scanned letter",
    "LEARNINGS FROM OTHER STATES":   "case study",
}

def read_side_label(page):
    """Return the rotated margin text, e.g. 'RECRUITMENT', or '' if none."""
    rotated = sorted((c for c in page.chars if not c["upright"]), key=lambda c: c["top"])
    return "".join(c["text"] for c in rotated).strip()


def classify_pdf_pages(pdf_path, form_pages=(), cover_pages=(1,)):
    rows = []                                    # list of dicts → DataFrame at the end

    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)

        # side labels read backwards on left-hand pages; the spelling that appears
        # forwards somewhere in the book is the real one
        raw_labels = [read_side_label(p) for p in pdf.pages]
        known = set(raw_labels)

        for page_no, page in enumerate(pdf.pages, start=1):
            side = raw_labels[page_no - 1]
            if side and side not in PART_LABELS and side[::-1] in known | set(PART_LABELS):
                side = side[::-1]                # 'TNEMTIURCER' → 'RECRUITMENT'

            upright_chars = [c for c in page.chars if c["upright"]]
            char_count  = len(upright_chars)
            image_count = len(page.images)
            shape_count = len(page.rects) + len(page.lines)
            text = (page.extract_text() or "").strip()

            # ---- rules, most reliable first ----
            if page_no in cover_pages:
                label = "cover"
            elif page_no == total_pages:
                label = "back cover"
            elif side:                                               # 1. side label decides the part
                label = PART_LABELS.get(side, "chapter body")
                if label.startswith("annexure I") and page_no in form_pages:
                    label = "annexure form"                          # manual list of form pages
            elif char_count == 0 and image_count == 0 and shape_count == 0:
                label = "blank"
            elif image_count > 0 and char_count < 200:
                label = "scanned message"
            elif text.lower() == "notes":
                label = "notes"
            else:
                label = "title / other"                              # check these by hand

            rows.append({
                "page_number": page_no,
                "label": label,
                "part": side or None,                                # e.g. 'RECRUITMENT': useful metadata later
                "chars_count": char_count,
                "images_count": image_count,
                "lines_and_rects_count": shape_count,
            })

    return pd.DataFrame(rows).set_index("page_number")





In [118]:
FORM_PAGES = {84, 85, 86, 87, 90, 91, 92, 93, 94, 95, 105, 106, 107, 108}   # manual decision
df_map_pdf = classify_pdf_pages(pdf_path, form_pages=FORM_PAGES)


In [119]:
df_map_pdf

,label,part,chars_count,images_count,lines_and_rects_count
page_number,,,,,
1,cover,NaN,0,1,494
2,blank,NaN,0,0,0
3,title / other,NaN,153,0,0
4,blank,NaN,0,0,0
5,scanned message,NaN,0,2,0
...,...,...,...,...,...
170,notes,NaN,5,0,29
171,notes,NaN,5,0,29
172,notes,NaN,5,0,29


In [57]:
df_map_pdf['label'].value_counts()

label
annexure I (procedures)    45
scanned letter             37
chapter body               29
case study                 24
annexure form              14
blank                       8
scanned message             5
contents                    4
notes                       3
abbreviations               2
cover                       1
title / other               1
back cover                  1
Name: count, dtype: int64

In [58]:
df_map_pdf.head(5)

,page_number,label,part,chars_count,images_count,lines_and_rects_count
0,1,cover,NaN,0,1,494
1,2,blank,NaN,0,0,0
2,3,title / other,NaN,153,0,0
3,4,blank,NaN,0,0,0
4,5,scanned message,NaN,0,2,0


## Step 3 · Page numbers — `page_num_verification()`
**Why:** the book's printed page (58) differs from the PDF page (78). Citations must show the number a reader can find.
**How:** compare two independent sources
- footer text below y = 780 (pdfplumber)
- page labels stored inside the PDF (pypdf)

**Result:** 155 Matched · 19 No Footer · **0 Mismatch**
**Decision:** `printed_page = footer or label`. Users see the printed page; the app opens the PDF page. Page numbers are metadata — never embedded.

In [59]:
#extarcting the page_number from the Footer_page 

page_number = []
#open the pdf
with pdfplumber.open(pdf_path) as pdf:

    #extarcting each  word
    for page in pdf.pages[:10]:
         words = page.extract_words() #Extracted eac charter by this Function
         Footer_word = [ w['text']
                        for w in words
                        if w.get("top",0) >  780 ]
         Footer_text = " ".join(Footer_word) if Footer_word   else None

    page_number.append(Footer_text)

print(page_number)


[None]


In [60]:
from pypdf import PdfReader

# 1. Open the document
reader = PdfReader(pdf_path)

# 2. Safely read and map the formal page labels
for idx, page in enumerate(reader.pages, start= 1):
    # Fetch the formal page label assigned to this index
    # page_labels is zero-based, while enumerate() starts at 1
    label = reader.page_labels[idx - 1] if idx - 1 < len(reader.page_labels) else "No label"
    
    print(f"Physical Page Index: {idx} ---> PDF Logical Page Label: {label}")


Physical Page Index: 1 ---> PDF Logical Page Label: i
Physical Page Index: 2 ---> PDF Logical Page Label: ii
Physical Page Index: 3 ---> PDF Logical Page Label: i
Physical Page Index: 4 ---> PDF Logical Page Label: ii
Physical Page Index: 5 ---> PDF Logical Page Label: iii
Physical Page Index: 6 ---> PDF Logical Page Label: iv
Physical Page Index: 7 ---> PDF Logical Page Label: v
Physical Page Index: 8 ---> PDF Logical Page Label: vi
Physical Page Index: 9 ---> PDF Logical Page Label: vii
Physical Page Index: 10 ---> PDF Logical Page Label: viii
Physical Page Index: 11 ---> PDF Logical Page Label: ix
Physical Page Index: 12 ---> PDF Logical Page Label: x
Physical Page Index: 13 ---> PDF Logical Page Label: xi
Physical Page Index: 14 ---> PDF Logical Page Label: xii
Physical Page Index: 15 ---> PDF Logical Page Label: xiii
Physical Page Index: 16 ---> PDF Logical Page Label: xiv
Physical Page Index: 17 ---> PDF Logical Page Label: xv
Physical Page Index: 18 ---> PDF Logical Page Label: 

In [61]:
import re
import pdfplumber
import pandas as pd
from pypdf import PdfReader

def page_num_verification(pdf_path, footer_top=780):
    """
    Build a page-number map for every page in the PDF.

    Source 1: the number printed in the footer (pdfplumber, words below `footer_top`)
    Source 2: the page label stored inside the PDF (pypdf)

    Returns a DataFrame with one row per page:
    pdf_page, footer_text, pdf_label, printed_page, status
    """
    # source 2: read all labels once, before the loop
    labels = PdfReader(pdf_path).page_labels

    page_num_map = []
    with pdfplumber.open(pdf_path) as nhm_pdf:              # closed automatically at the end
        if len(labels) != len(nhm_pdf.pages):
            raise ValueError(f"Page count differs: pypdf {len(labels)} vs pdfplumber {len(nhm_pdf.pages)}")

        for pdf_page_no, page in enumerate(nhm_pdf.pages, start=1):
            # source 1: words in the footer strip
            footer_words = [w["text"] for w in page.extract_words() if w["top"] > footer_top]
            footer_text = " ".join(footer_words) if footer_words else None

            # safety: a footer must look like a page number (58) or a roman numeral (xiii)
            if footer_text and not re.fullmatch(r"\d+|[ivxlcdm]+", footer_text, re.IGNORECASE):
                footer_text = None

            pdf_label = labels[pdf_page_no - 1]

            if footer_text is None:
                status = "No Footer"
            elif footer_text == pdf_label:
                status = "Matched"
            else:
                status = "Mismatch"

            page_num_map.append({
                "pdf_page": pdf_page_no,
                "footer_text": footer_text,
                "pdf_label": pdf_label,
                "printed_page": footer_text or pdf_label,     # footer first, label as fallback
                "status": status,
            })

    return pd.DataFrame(page_num_map)

In [62]:
df_page_num = page_num_verification(pdf_path)
df_page_num.head(20)

,pdf_page,footer_text,pdf_label,printed_page,status
0,1,NaN,i,i,No Footer
1,2,NaN,ii,ii,No Footer
2,3,NaN,i,i,No Footer
3,4,NaN,ii,ii,No Footer
4,5,NaN,iii,iii,No Footer
5,6,NaN,iv,iv,No Footer
6,7,NaN,v,v,No Footer
7,8,NaN,vi,vi,No Footer
8,9,NaN,vii,vii,No Footer
9,10,NaN,viii,viii,No Footer


In [63]:
df_page_num["status"].value_counts()                               # expect: mostly Matched, some No Footer, 0 Mismatch
    # spot-check key pages

status
Matched      155
No Footer     19
Name: count, dtype: int64

In [64]:
df_page_num[df_page_num["status"] == "Mismatch"]                   # should be empty


,pdf_page,footer_text,pdf_label,printed_page,status


In [65]:
df_page_num[df_page_num["status"] == "No Footer"]    # returns nothing because of the space

,pdf_page,footer_text,pdf_label,printed_page,status
0,1,NaN,i,i,No Footer
1,2,NaN,ii,ii,No Footer
2,3,NaN,i,i,No Footer
3,4,NaN,ii,ii,No Footer
4,5,NaN,iii,iii,No Footer
5,6,NaN,iv,iv,No Footer
6,7,NaN,v,v,No Footer
7,8,NaN,vi,vi,No Footer
8,9,NaN,vii,vii,No Footer
9,10,NaN,viii,viii,No Footer


## Step 4 · Look at sample pages (manual)
**Why:** eyes catch what code doesn't flag.
**Found:**
- Forms (travel, advance, exit, appraisal) — templates, not real tables
- Flowchart on PDF 32 — text comes out jumbled
- PDF 90 — leftover sample text *"Air tickets booked by myself…"*

**Decision:** recorded in the manual decisions list.

## Step 5 · Pages with real text — `pages_with_real_text()`
**Why:** profile only pages whose text is real — scanned pages would distort the numbers.
**How:** ≥ 200 **upright** characters and images < 50 % of the page. 200 sits in a natural gap (nothing between 123 and 261).
**Result:** **118 pages** (15–108, 146–169) — identical to the step-2 labels ✅
**Note — why `upright`:** the rotated side label is decoration. On near-empty pages it dominates (PDF 113: 24 chars, 21 are the label).

In [66]:
import pdfplumber

def pages_with_real_text(pdf_path, min_chars: int = 200, max_image_share: float = 0.5):
    """
    Return the PDF page numbers (1-based) that contain real, selectable text.

    A page counts if:
      - it has at least `min_chars` upright characters (the rotated side label is ignored), and
      - images cover less than `max_image_share` of the page (0.5 = half the page).
    """
    result = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_no, page in enumerate(pdf.pages, start=1):
            upright_chars = [c for c in page.chars if c["upright"]]
            n_chars = len(upright_chars)

            page_area = page.width * page.height
            image_area = sum(img["width"] * img["height"] for img in page.images)
            image_share = image_area / page_area

            if n_chars >= min_chars and image_share < max_image_share:
                result.append(page_no)
    return result

In [67]:
real_pages = pages_with_real_text(pdf_path)                  # uses the defaults 200 and 0.5

In [68]:
print(len(real_pages))                    # 118
print(real_pages[:3], real_pages[-3:])    # [15, 16, 17] [167, 168, 169]

118
[15, 16, 17] [167, 168, 169]


In [69]:
must_be_in  = [15, 21, 90, 108, 146, 169]    # abbreviations, chapter 1, a form, last annexure I page, first case study, last case study
must_be_out = [3, 5, 109, 118, 170, 174]     # title, message, scanned letters, notes, back cover

print(all(p in real_pages for p in must_be_in))        # True
print(any(p in real_pages for p in must_be_out))       # False

True
False


In [70]:
jumps = [(a, b) for a, b in zip(real_pages, real_pages[1:]) if b != a + 1]
print(jumps)                                           # [(108, 146)]

[(108, 146)]


In [71]:
for t in (150, 200, 250):
    print(t, len(pages_with_real_text(pdf_path, min_chars=t)))
# 150 → 119  (page 3, the title page, sneaks in with 153 chars)
# 200 → 118
# 250 → 118   ← same as 200, so 200 is a safe choice

150 119
200 118
250 118


In [72]:
SKIP = {
    "cover",
    "blank",
    "title / other",
    "contents",
    "scanned message",
    "scanned letter",
    "notes",
    "back cover",
}

label_pages = df_map_pdf.loc[
    ~df_map_pdf["label"].isin(SKIP), "page_number"
].tolist()

print(set(real_pages) == set(label_pages))  # True
print(set(real_pages) ^ set(label_pages))   # set()
print(set(real_pages) == set(label_pages))             # True
print(set(real_pages) ^ set(label_pages))              # set() → no disagreements

False
{17, 18, 19, 20}
False
{17, 18, 19, 20}


In [73]:
with pdfplumber.open(pdf_path) as pdf:
    for page_no in real_pages:
        page = pdf.pages[page_no - 1]
        ...  # font profile

In [74]:
print("total count of real charter pages area ", len(real_pages))

total count of real charter pages area  118


## Step 6 · Font profile — `font_profiling()`
**Why:** font size + weight tell us what each line *is* (heading, body, footnote).
**Result:**

| Style | Meaning |
|---|---|
| Regular 11 (~91 %) | body text |
| Bold 30 | chapter / annexure title (level 0) |
| Bold 14 | section heading (level 1) |
| Semibold 13 | subsection (level 2) |
| Semibold 12 | exhibit / sub-subheading (level 3) |
| Bold 11 | bold words inside body — **not** a heading |
| Regular 8 | footnote |
| < 7.5 | superscript |

**Decision:** heading = size ≥ 12 **and** bold/semibold.

In [75]:
def font_profiling(pdf_path, page_numbers):

    """ this  function profiling  the font types there size """
    stats = {}                                          # (font, size) → counts
   #open the pdf
    with pdfplumber.open(pdf_path) as pdf:
        for page_no in page_numbers:
            # page_numbers contains the REAL PDF page numbers
             # Example: [21, 22, 25, 30]
            page = pdf.pages[page_no - 1]

            for line in page.extract_text_lines(return_chars=True):
                for char in line["chars"]:
                    if not char["upright"]:   #skipping upright character
                        continue
                    
                    # Remove the prefix from font name
                    # Example:
                    # EUUBGP+ProximaNovaA-Bold
                    # becomes:
                    # ProximaNovaA-Bold
                    font = char["fontname"].split("+")[-1]
                    size = round(char["size"], 1) # Round font size
                    key = (font, size)

                    # If we haven't seen this font/size before
                    if key not in stats:                # ✅ not in
                        stats[key] = {"chars": 0, "pages": set(), "example": None}

                    stats[key]["chars"] += 1
                    stats[key]["pages"].add(page_no)
                    if stats[key]["example"] is None:
                        stats[key]["example"] = (page_no, line["text"][:60])

    total_chars = sum(item["chars"] for item in stats.values())
    
    # Convert dictionary into DataFrame
    rows = []
    for (font, size), info in stats.items():
        pages = sorted(info["pages"])
        rows.append({                                   # ✅ inside the loop
            "font": font,
            "size": size,
            "chars": info["chars"],
            "percent": round(info["chars"] / total_chars * 100, 2),
            "n_pages": len(pages),
            "pages": pages,
            "example": info["example"],
        })

    df = pd.DataFrame(rows)
    return df.sort_values("chars", ascending=False).reset_index(drop=True)

In [76]:
df_font_profiling  = font_profiling(pdf_path,real_pages)
df_font_profiling

,font,size,chars,percent,n_pages,pages,example
0,ProximaNovaA-Regular,11.0,181188,90.87,118,"[15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 2...","(15, ANM Auxiliary Nurse Midwife)"
1,MyriadPro-Regular,11.0,4591,2.30,4,"[17, 18, 19, 20]","(17, Chapter 1. Introduction 1-2)"
2,ProximaNovaA-Bold,11.0,2803,1.41,29,"[47, 50, 51, 52, 53, 75, 77, 78, 82, 83, 84, 8...","(47, • HR Planning- It includes information th..."
3,ProximaNovaA-Bold,14.0,2438,1.22,69,"[21, 22, 23, 25, 26, 28, 29, 30, 31, 32, 37, 3...","(21, Chapter)"
4,ProximaNovaA-Regular,8.0,1772,0.89,13,"[21, 23, 24, 26, 32, 35, 45, 46, 47, 49, 78, 1...","(21, 1 Ministry of Health and Family Welfare, ..."
5,ProximaNovaA-Semibold,13.0,1763,0.88,42,"[23, 24, 32, 33, 34, 35, 37, 38, 39, 40, 43, 4...","(23, 2.1.1. Service Delivery)"
6,ProximaNovaA-Semibold,12.0,1115,0.56,22,"[32, 34, 37, 45, 54, 56, 58, 60, 64, 66, 67, 6...","(32, Exhibit 1: Key steps in the Process of Re..."
7,MetaPro,11.0,842,0.42,6,"[148, 149, 162, 163, 165, 167]","(148, – The Masters: includes the updated Huma..."
8,ProximaNovaA-RegularIt,11.0,780,0.39,7,"[27, 35, 46, 48, 73, 87, 102]","(27, Note: All HR related policies of the Stat..."
9,ProximaNovaA-Bold,30.0,738,0.37,22,"[15, 17, 21, 23, 25, 28, 31, 37, 39, 42, 48, 5...","(15, List of Abbreviations)"


In [77]:
df_font_profiling.loc[0, ["font", "size"]].tolist()  

['ProximaNovaA-Regular', np.float64(11.0)]

In [78]:
df_font_profiling["font"].str.contains(r"\+").any() 

np.False_

## Step 7 · Clutter inventory — `clutter_inventory()`
**Why:** list every kind of non-content before removing anything. Each row becomes one cleaning rule.
**Result & decision:**

| Clutter | Count | Decision |
|---|---|---|
| side label | 118 | drop, save as `part` |
| page number | 118 | drop (saved as `printed_page`) |
| fill-in `___` / `……` | 76 | forms → drop · body → `[__]` |
| bullet `y` | 65 | → `- ` |
| ordinal ᵗʰ | 44 | drop |
| footnote text | 24 | move to `footnotes` |
| footnote marker ¹ | 15 | drop |
| repeated lines | 3 | not clutter (real headings) |

**Checkpoint later:** re-run on clean text → everything marked "drop" must be 0.

In [79]:
import re
from collections import Counter
import pdfplumber
import pandas as pd

def clutter_inventory(pdf_path, page_numbers, footer_top=780, footnote_zone=650,
                      superscript_max=7.5, footnote_size=8.5, repeat_min_pages=5):
    """Find and count every kind of clutter on the given pages. Nothing is removed."""
    inventory = {}                       # clutter_type -> {"count", "pages", "example"}
    line_seen_on = {}                    # line text    -> set of pages (for repeated lines)

    # ---------- the helper ----------
    def add(clutter_type, page_no, example_text):
        # create the entry the first time we see this clutter type
        if clutter_type not in inventory:
            inventory[clutter_type] = {"count": 0, "pages": set(), "example": None}
        inventory[clutter_type]["count"] += 1
        inventory[clutter_type]["pages"].add(page_no)
        if inventory[clutter_type]["example"] is None:
            inventory[clutter_type]["example"] = (page_no, example_text)

    with pdfplumber.open(pdf_path) as pdf:
        for page_no in page_numbers:
            page = pdf.pages[page_no - 1]

            # 1. side label: rotated characters, read top to bottom
            rotated = sorted((c for c in page.chars if not c["upright"]), key=lambda c: c["top"])
            side_label = "".join(c["text"] for c in rotated)
            if side_label:
                add("side label", page_no, side_label)

            # 2. page number: words below the footer line
            footer = " ".join(w["text"] for w in page.extract_words() if w["top"] > footer_top)
            if footer:
                add("page number", page_no, footer)

            # 3 & 4. character-level clutter
            for char in page.chars:
                if char["upright"] and char["size"] < superscript_max and char["top"] <= footer_top:
                    if char["text"].isdigit():
                        add("footnote marker", page_no, char["text"])     # health¹
                    else:
                        add("ordinal suffix", page_no, char["text"])      # 19ᵗʰ
                if "Wingdings" in char["fontname"]:
                    add("bullet glyph", page_no, char["text"])

            # 5 & 6. line-level clutter
            for line in page.extract_text_lines(return_chars=True):
                upright = [c for c in line["chars"] if c["upright"]]
                if not upright:
                    continue
                size = Counter(round(c["size"], 1) for c in upright).most_common(1)[0][0]
                text = line["text"].strip()

                if size <= footnote_size and footnote_zone < line["top"] <= footer_top:
                    add("footnote text", page_no, text[:60])
                if re.search(r"_{3,}|…{2,}|\.{4,}", text):
                    add("fill-in line", page_no, text[:60])
                if line["top"] <= footer_top:
                    line_seen_on.setdefault(text, set()).add(page_no)

    # 7. repeated lines: same text on many pages (running headers / footers)
    for text, pages in line_seen_on.items():
        if len(pages) >= repeat_min_pages:
            add("repeated line", min(pages), f"{text[:50]} (on {len(pages)} pages)")

    rows = [{"clutter_type": k, "count": v["count"], "n_pages": len(v["pages"]),
             "example": v["example"]} for k, v in inventory.items()]
    return pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)

In [80]:
clutter_df = clutter_inventory(pdf_path, real_pages)
clutter_df

,clutter_type,count,n_pages,example
0,side label,118,118,"(15, LIST OF ABBREVIATIONS)"
1,page number,118,118,"(15, xiii)"
2,fill-in line,76,15,"(66, facilities in the district within ___ day..."
3,bullet glyph,65,14,"(21, y)"
4,ordinal suffix,44,3,"(19, t)"
5,footnote text,24,13,"(21, 1 Ministry of Health and Family Welfare, ..."
6,footnote marker,15,10,"(21, 1)"
7,repeated line,3,2,"(21, Chapter (on 9 pages))"


In [81]:
set(clutter_df["clutter_type"])          # all 8 types present


{'bullet glyph',
 'fill-in line',
 'footnote marker',
 'footnote text',
 'ordinal suffix',
 'page number',
 'repeated line',
 'side label'}

In [82]:

clutter_df.set_index("clutter_type").loc["side label", "count"] == len(real_pages)    # True: one per page


np.True_

In [83]:

clutter_df.set_index("clutter_type").loc["page number", "count"] == len(real_pages)   # True: one per page

np.True_

## Step 8 · Table inventory — `table_inventory()`
**Why:** tables break most easily — know them before extracting.
**Result:** 36 tables on 24 pages → 19 data · 15 form grids · 2 lists
- Continue onto next page (copy header): 16, 86, 100, 102, 106, 108
- Fixed by hand: PDF 101 has a 3-row header · 94–95 are form grids

**Decision:** data table → Markdown · form grid → field list · 2-col list → glossary

In [84]:
import re
import pdfplumber
import pandas as pd

def table_inventory(pdf_path, page_numbers):
    rows = []
    prev = None                                        # last table seen, for the continuation check

    with pdfplumber.open(pdf_path) as pdf:
        for page_no in page_numbers:
            page = pdf.pages[page_no - 1]

            for table_no, table in enumerate(page.find_tables(), start=1):
                # ---- everything below is INSIDE the table loop ----

                cells = [[(cell or "").replace("\n", " ").strip() for cell in row] for row in table.extract()]

                n_rows = len(cells)
                n_cols = max((len(row) for row in cells), default=0)

                total_cells = n_rows * n_cols
                non_empty = sum(1 for row in cells for cell in row if cell != "")
                fill_percentage = non_empty / total_cells * 100 if total_cells else 0

                body = cells[1:]                                         # cells, not cell
                body_non_empty = sum(1 for row in body for cell in row if cell != "")

                top, bottom = table.bbox[1], table.bbox[3]

                # ---- table type ----
                if non_empty == 0:
                    kind = "decorative / empty"
                elif body_non_empty == 0 or fill_percentage < 35:
                    kind = "form grid"
                elif n_cols == 2 and all(len(row[0]) <= 8 for row in cells if row and row[0]):
                    kind = "2-col list"
                else:
                    kind = "data table"

                # ---- header over several rows? ----
                multi_row_header = False
                if n_rows > 2:
                    first_row, second_row = cells[0], cells[1]
                    first_has_empty = any(cell == "" for cell in first_row)
                    second_has_text = any(cell != "" for cell in second_row)
                    second_no_numbers = not any(re.search(r"\d", cell) for cell in second_row)
                    multi_row_header = first_has_empty and second_has_text and second_no_numbers

                # ---- continues from previous page? ----
                continues_prev = (
                    table_no == 1
                    and top < 120
                    and prev is not None
                    and prev["page"] == page_no - 1
                    and prev["bottom"] > 700
                    and prev["n_cols"] == n_cols
                )

                rows.append({
                    "page": page_no,
                    "table_no": table_no,
                    "n_rows": n_rows,
                    "n_cols": n_cols,
                    "fill_percent": round(fill_percentage, 2),
                    "kind": kind,
                    "multi_row_header": multi_row_header,
                    "continues_prev": continues_prev,
                    "top": round(top, 1),
                    "bottom": round(bottom, 1),
                })

                prev = {"page": page_no, "bottom": bottom, "n_cols": n_cols}

    return pd.DataFrame(rows)

In [85]:
df_table = table_inventory(pdf_path,real_pages)

In [86]:
df_table

,page,table_no,n_rows,n_cols,fill_percent,kind,multi_row_header,continues_prev,top,bottom
0,15,1,19,2,100.00,2-col list,False,False,322.2,760.9
1,16,1,16,2,100.00,2-col list,False,True,54.8,424.6
2,34,1,7,4,75.00,data table,False,False,256.1,386.7
3,52,1,16,2,87.50,data table,True,False,285.8,748.8
4,76,1,4,2,50.00,data table,True,False,163.1,229.9
5,78,1,14,3,90.48,data table,False,False,78.6,522.2
6,82,1,4,4,81.25,data table,False,False,72.6,216.9
7,84,1,5,3,20.00,form grid,False,False,338.4,421.8
8,84,2,5,2,20.00,form grid,False,False,638.8,722.3
9,85,1,3,1,100.00,data table,False,False,99.2,357.3


In [87]:
df_table['kind'].value_counts()

kind
data table    19
form grid     15
2-col list     2
Name: count, dtype: int64

In [88]:
len(df_table)

36

## Step 9 · Images & diagrams — `image_inventory()` + `diagram_inventory()`
**Why:** find everything that isn't plain text.
**Result:**
- 44 images → 43 scanned + 1 tiny dot. No photos or charts.
- **42 OCR pages** (5 messages + 37 letters) — matches step-2 labels ✅
- Only non-table drawing: PDF 32 flowchart

**Decision:** OCR the 42 pages · write the flowchart's 8 steps by hand.

In [89]:
import pdfplumber
import pandas as pd

def image_inventory(pdf_path, tiny_share=0.01, min_chars=200):
    rows = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_no, page in enumerate(pdf.pages, start=1):
            page_area = page.width * page.height
            upright_chars = sum(1 for c in page.chars if c["upright"])

            for image_no, img in enumerate(page.images, start=1):
                img_share = (img["width"] * img["height"]) / page_area

                if img_share < tiny_share:
                    verdict = "tiny / decoration"
                elif upright_chars < min_chars:
                    verdict = "scanned page"
                else:
                    verdict = "picture inside text"

                rows.append({                                   # ( on the same line
                    "page_number": page_no,                     # the number, not the Page object
                    "image_no": image_no,
                    "image_share": round(img_share, 2),
                    "upright_chars": upright_chars,
                    "verdict": verdict,
                })

    return pd.DataFrame(rows)

In [90]:
df_image_inventory = image_inventory(pdf_path)


In [91]:
df_image_inventory

,page_number,image_no,image_share,upright_chars,verdict
0,1,1,0.00,0,tiny / decoration
1,5,1,0.82,0,scanned page
2,5,2,1.02,0,scanned page
3,7,1,0.82,0,scanned page
4,9,1,0.82,0,scanned page
5,11,1,0.82,0,scanned page
6,13,1,0.82,0,scanned page
7,109,1,0.54,123,scanned page
8,110,1,0.68,63,scanned page
9,111,1,0.59,88,scanned page


In [92]:
df_image_inventory['verdict'].value_counts()

verdict
scanned page         43
tiny / decoration     1
Name: count, dtype: int64

In [93]:

ocr_pages = sorted(df_image_inventory.loc[df_image_inventory["verdict"] == "scanned page", "page_number"].unique().tolist())

print(len(ocr_pages))    # 42 [1] [32]

42


In [94]:
print(type(ocr_pages))        # <class 'list'>
print(type(ocr_pages[0]))     # <class 'int'>
print(len(ocr_pages))         # 42
print(ocr_pages[:5], ocr_pages[-3:])   # [5, 7, 9, 11, 13] [143, 144, 145]

<class 'list'>
<class 'int'>
42
[5, 7, 9, 11, 13] [143, 144, 145]


In [110]:
def diagram_inventory(pdf_path, page_numbers, margin_left=60, margin_right=50, min_width=40, min_height=15, footer_top=780):

    def _inside_any(obj, boxes):
        """Check whether the center point of an object falls inside any table bounding box."""
        x = (obj["x0"] + obj["x1"]) / 2
        y = (obj["top"] + obj["bottom"]) / 2
        return any(x0 <= x <= x1 and top <= y <= bottom for x0, top, x1, bottom in boxes)

    rows = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_no in page_numbers:
            page = pdf.pages[page_no - 1]

            # Get bounding boxes of tables so their shapes can be ignored
            boxes = [t.bbox for t in page.find_tables()]

            shapes = []

            for shape in page.rects + page.curves:
                # Skip shapes inside tables
                if _inside_any(shape, boxes):
                    continue

                # Skip shapes near the left/right page margins
                if shape["x0"] < margin_left or shape["x1"] > page.width - margin_right:
                    continue

                # Skip very small shapes
                if shape["width"] < min_width or shape["height"] < min_height:
                    continue

                # Skip header/footer area
                if shape["top"] < 60 or shape["top"] > footer_top:
                    continue

                # Keep the shape if it passed all filters
                shapes.append(shape)

            # After checking all shapes, store information if any were found
            if shapes:
                first_three = []

                for shape in shapes[:3]:
                    first_three.append({
                        "x0": round(shape["x0"], 1),
                        "top": round(shape["top"], 1),
                        "width": round(shape["width"], 1),
                        "height": round(shape["height"], 1)
                    })

                rows.append({
                    "page": page_no,
                    "n_shapes": len(shapes),
                    "first_3_positions": first_three
                })

    return pd.DataFrame(rows)

In [113]:
df_diagram_inventory = diagram_inventory(pdf_path,real_pages)
df_diagram_inventory

,page,n_shapes,first_3_positions
0,32,16,"[{'x0': 94.7, 'top': 486.9, 'width': 113.4, 'h..."


In [116]:
ocr_pages

[5,
 7,
 9,
 11,
 13,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145]

In [115]:
df_map_pdf[df_map_pdf["label"].isin(["scanned message", "scanned letter"])].index.tolist()

[4,
 6,
 8,
 10,
 12,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144]

In [120]:
scanned_by_label = df_map_pdf[df_map_pdf["label"].isin(["scanned message", "scanned letter"])].index.tolist()
set(ocr_pages) == set(scanned_by_label)      # True

True

## Step 10 · Special characters — `special_char_inventory()`
**Why:** typeset characters (`’`, `–`) don't match what users type (`'`, `-`).
**Result & decision:** `•`→`- ` · `’‘`→`'` · `“”`→`"` · `–`→`-` · `—`→` - ` · `√`→`Yes` · `…` → fill-in rule
**Notes:**
- Ligatures (`fi`, `fl`) appear **twice** in `line["chars"]` → cause of the old `offificial` bug. Use `line["text"]`.
- No `(cid:)` errors · no Hindi in the text layer (only in scans)

In [121]:
import re
import unicodedata
from collections import Counter
import pdfplumber
import pandas as pd

def special_char_inventory(pdf_path, page_numbers):
    """Find every character outside plain keyboard text (ASCII 32–126). Nothing is changed."""
    inventory = {}            # character -> {"count", "pages", "example"}
    ligatures = Counter()     # char objects holding several letters, e.g. "fi"
    cid_count = 0             # "(cid:NN)" = a character the font could not decode

    # ---------- helper (same pattern as the clutter inventory) ----------
    def add(char, page_no, line_text):
        if char not in inventory:
            inventory[char] = {"count": 0, "pages": set(), "example": None}
        inventory[char]["count"] += 1
        inventory[char]["pages"].add(page_no)
        if inventory[char]["example"] is None:
            inventory[char]["example"] = (page_no, line_text[:60])

    with pdfplumber.open(pdf_path) as pdf:
        for page_no in page_numbers:
            page = pdf.pages[page_no - 1]

            # 1. broken fonts: pdfplumber writes "(cid:NN)" when it can't decode a character
            cid_count += len(re.findall(r"\(cid:\d+\)", page.extract_text() or ""))

            for line in page.extract_text_lines(return_chars=True):
                seen = set()                             # ligatures appear twice in line["chars"]
                for char in line["chars"]:
                    if not char["upright"]:              # skip the side label
                        continue
                    position = (round(char["x0"], 1), round(char["top"], 1), char["text"])
                    if position in seen:                 # same char at the same spot = duplicate
                        continue
                    seen.add(position)

                    # 2. ligatures: one char object, several letters
                    if len(char["text"]) > 1:
                        ligatures[char["text"]] += 1

                    # 3. special characters
                    for letter in char["text"]:
                        code = ord(letter)
                        if code < 32 or code > 126:     # outside plain keyboard text
                            add(letter, page_no, line["text"])

    # ---------- turn into a table ----------
    rows = []
    for char, info in inventory.items():
        rows.append({
            "char": char,
            "code": hex(ord(char)),
            "name": unicodedata.name(char, "UNKNOWN"),
            "count": info["count"],
            "n_pages": len(info["pages"]),
            "example": info["example"],
        })
    df = pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)
    return df, dict(ligatures), cid_count

In [122]:
special_df, ligatures, cid_count = special_char_inventory(pdf_path, real_pages)
special_df

,char,code,name,count,n_pages,example
0,•,0x2022,BULLET,556,50,"(26, • Ensure availability of HR by creating a..."
1,…,0x2026,HORIZONTAL ELLIPSIS,370,7,"(75, This agreement is made on ………………………, BETW..."
2,’,0x2019,RIGHT SINGLE QUOTATION MARK,55,35,"(15, ESI Employee’s State Insurance)"
3,–,0x2013,EN DASH,48,18,"(45, and evaluated at regular intervals. The i..."
4,√,0x221a,SQUARE ROOT,12,2,"(34, Written Test √ √ √)"
5,‘,0x2018,LEFT SINGLE QUOTATION MARK,9,7,"(29, The ‘deputation to SHS’ is considered as ..."
6,“,0x201c,LEFT DOUBLE QUOTATION MARK,6,6,"(21, The vision of NHM is: “Attainment of Univ..."
7,”,0x201d,RIGHT DOUBLE QUOTATION MARK,6,6,"(21, convergent action to address the wider so..."
8,—,0x2014,EM DASH,2,2,"(20, Part 2 : Recruiting and Retaining Special..."


In [123]:
print(ligatures) 

{'ff': 4, 'ffi': 3, 'fi': 471, 'fl': 18}


In [124]:
print(cid_count)  

0


In [125]:
special_df["name"].str.startswith("DEVANAGARI").any()     # False → no Hindi in the text layer

np.False_

## Step 11 · Table of contents — `read_contents_lines()` + `parse_contents()`
**Why:** the Contents is the answer key — it lists every section and its start page.
**How:**
- An entry ends when its text ends with a page number (joins wrapped lines)
- Level from the numbering pattern (`5.3.4.` → level 2), not the indent

**Result:** 127 headings → 59 sections · 28 annexure items · 21 letters · 9 chapters · 7 case-study parts · 3 annexures
**Notes:** no `2.1` / `7.1` in the Contents · letters get their title and date from here (their text comes from OCR).
**Decision:** used in Stage E to place body text under the right section (TOC-guided sectioning).

In [126]:
import re
import pdfplumber
import pandas as pd

# An entry is complete when it ENDS with a page number ("14") or a range ("11-16")
PAGE_AT_END = re.compile(r"^(?P<title>.*?)\s+(?P<start>\d+)(?:-(?P<end>\d+))?$")


def read_contents_lines(pdf_path, contents_pages, footer_top=780):
    """Read the Contents pages and return one string per entry (wrapped lines joined)."""
    entries = []
    buffer = ""                                    # holds an entry that isn't finished yet

    with pdfplumber.open(pdf_path) as pdf:
        for page_no in contents_pages:
            page = pdf.pages[page_no - 1]
            page = page.filter(lambda obj: obj.get("upright", True))   # drop the rotated side label

            for line in page.extract_text_lines():
                text = line["text"].strip()

                if line["top"] > footer_top or text.upper() == "CONTENTS":
                    continue                       # skip page number and the "CONTENTS" title

                buffer = f"{buffer} {text}".strip()        # add this line to the unfinished entry

                if PAGE_AT_END.match(buffer):              # ends with a page number → entry complete
                    entries.append(buffer)
                    buffer = ""

    if buffer:                                     # leftover text = an entry that never ended (check!)
        entries.append(buffer)
    return entries


def parse_contents(entries):
    """Turn each entry into: level, kind, number, title, printed_start, printed_end, parent."""
    rows = []
    parent = None                                  # current chapter / annexure

    for entry in entries:
        m = PAGE_AT_END.match(entry)
        title = m["title"]
        start = int(m["start"])
        end = int(m["end"]) if m["end"] else start

        if mm := re.match(r"^Chapter (\d+)\.\s*(.*)", title):                 # Chapter 5. Recruitment
            level, kind, number, title = 0, "chapter", mm[1], mm[2]
        elif mm := re.match(r"^Annexure ([IVX]+)\.\s*(.*)", title):           # Annexure II. Compendium...
            level, kind, number, title = 0, "annexure", mm[1], mm[2]
        elif mm := re.match(r"^(\d+(?:\.\d+)+)\.\s*(.*)", title):              # 5.3.4. Stages of Selection
            number, title = mm[1], mm[2]
            level, kind = number.count("."), "section"                         # "5.3.4" has 2 dots → level 2
        elif mm := re.match(r"^(\d+)\.\s*(Dated:.*)", title):                  # 5. Dated: 8th March...
            level, kind, number, title = 1, "letter", mm[1], mm[2]
        elif mm := re.match(r"^Part (\d+)\s*:\s*(.*)", title):                 # Part 1 : ...
            level, kind, number, title = 2, "case study part", mm[1], mm[2]
        else:                                                                  # Leave Policy
            level, kind, number = 1, "annexure item", None

        if level == 0:
            parent = f"{kind.title()} {number}"                                # "Chapter 5", "Annexure II"

        rows.append({
            "level": level,
            "kind": kind,
            "number": number,
            "title": title.strip(),
            "printed_start": start,
            "printed_end": end,
            "parent": parent,
        })

    return pd.DataFrame(rows)

In [135]:
CONTENTS_PAGES = [17, 18, 19, 20]
entries = read_contents_lines(pdf_path, CONTENTS_PAGES)
toc_df = parse_contents(entries)

print(len(entries), len(toc_df))     # 127 127
toc_df.head(10)

127 127


,level,kind,number,title,printed_start,printed_end,parent
0,0,chapter,1,Introduction,1,2,Chapter 1
1,1,section,1.1,Purpose of the Guidelines,2,2,Chapter 1
2,0,chapter,2,Human Resource for Health under NHM,3,4,Chapter 2
3,2,section,2.1.1,Service Delivery,3,3,Chapter 2
4,2,section,2.1.2,Programme Management,3,3,Chapter 2
5,2,section,2.1.3,Support Service,4,4,Chapter 2
6,2,section,2.1.4,Other HR,4,4,Chapter 2
7,0,chapter,3,"HRH Policy, Principles and Strategies",5,7,Chapter 3
8,1,section,3.1,Human Resource Principles in NHM,5,5,Chapter 3
9,1,section,3.2,Key Strategies,6,6,Chapter 3


In [139]:
len(toc_df)                                                  # 127
toc_df["kind"].value_counts()                                # section 59, annexure item 28, letter 21,
                                                             # chapter 9, case study part 7, annexure 3
toc_df["title"].str.contains("CONTENTS|STNETNOC").any()      # False → side label didn't leak
toc_df["printed_start"].is_monotonic_increasing              # True  → entries are in book order
toc_df.groupby("parent", sort=False).size()                  # headings per chapter/annexure

parent
Chapter 1        2
Chapter 2        5
Chapter 3        3
Chapter 4        8
Chapter 5       12
Chapter 6        7
Chapter 7        8
Chapter 8       19
Chapter 9        4
Annexure I      20
Annexure II     22
Annexure III    17
dtype: int64

In [142]:
printed_to_pdf = dict(zip(df_page_num["printed_page"], df_page_num["pdf_page"]))

toc_df["pdf_start"] = toc_df["printed_start"].astype(str).map(printed_to_pdf)

toc_df.loc[toc_df["number"] == "5.3.4", ["title", "printed_start", "pdf_start"]]
# Stages of Selection | 14 | 34

,title,printed_start,pdf_start
25,Stages of Selection,14,34


In [143]:
toc_df["pdf_start"].isna().sum()      # 0

np.int64(0)

## Lessons learned
| Symptom | Cause | Habit |
|---|---|---|
| wrong page's text | used a global `page` / `reader` | use the function's own parameter; restart & run all |
| 118 rows instead of 36 | `rows.append` at the wrong indentation | check `len(df)` after every function |
| every count = 1 | `is not` instead of `not in` | `not in` = membership |
| empty DataFrame, no error | `rows.append` and `(` on different lines | `(` on the same line as the call |
| `np.int64(5)` in lists | pandas numbers | `.tolist()` before saving JSON |

In [146]:
import json
from pathlib import Path
import os

ROOT = Path(pdf_path).resolve().parents[2]
PROFILE_DIR = ROOT / "data" / "profile"

page_map_df = df_map_pdf.reset_index()
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

page_map_df.to_csv(PROFILE_DIR / "page_map.csv", index=False)
df_table.to_csv(PROFILE_DIR / "table_inventory.csv", index=False)
toc_df.to_csv(PROFILE_DIR / "toc.csv", index=False)
clutter_df.to_csv(PROFILE_DIR / "clutter_before.csv", index=False)
special_df.to_csv(PROFILE_DIR / "special_chars_before.csv", index=False)

decisions = {
    "footer_top": 780, "footnote_zone": 650, "footnote_size": 8.5,
    "superscript_max": 7.5, "min_chars": 200,
    "cover_pages": [1],
    "contents_pages": [17, 18, 19, 20],
    "glossary_pages": [15, 16],
    "form_pages": [84, 85, 86, 87, 90, 91, 92, 93, 94, 95, 105, 106, 107, 108],
    "ocr_pages": ocr_pages,                      # plain ints, thanks to .tolist()
    "diagram_text": {"32": "Exhibit 1: Key steps in the Process of Recruitment — 1. Identify and validate vacancies → ..."},
    "oddities": {"90": "Air tickets booked by myself of both tickets Hyderabad"},
    "char_replacements": {"’": "'", "‘": "'", "“": '"', "”": '"', "–": "-", "—": " - ", "•": "- ", "√": "Yes"},
}
config_dir = ROOT / "config"
config_dir.mkdir(parents=True, exist_ok=True)

with open(config_dir / "manual_decisions.json", "w", encoding="utf-8") as f:
    json.dump(decisions, f, ensure_ascii=False, indent=2)